# ABSA MBG — Aspect-Based Sentiment Analysis
## Attention-Enhanced XGBoost + MLflow Tracking

Pipeline lengkap:
1. Load & validasi data
2. EDA (Exploratory Data Analysis)
3. Preprocessing teks (Bahasa Indonesia)
4. Pelabelan sentimen & aspek
5. Rekayasa fitur (TF-IDF + Attention)
6. SMOTE (handle class imbalance)
7. Training Attention-Enhanced XGBoost
8. Evaluasi (accuracy, F1, CV, confusion matrix)
9. Logging metrik & artefak ke MLflow
10. Simpan model & komponen ke disk


## 0. Install Dependensi

In [1]:
import subprocess
import sys
import importlib

REQUIRED_PACKAGES = {
    "xgboost"         : "xgboost",
    "PySastrawi"      : "Sastrawi",
    "nltk"            : "nltk",
    "scikit-learn"    : "sklearn",
    "mlflow"          : "mlflow",
    "imbalanced-learn": "imblearn",
    "wordcloud"       : "wordcloud",
    "seaborn"         : "seaborn",
    "matplotlib"      : "matplotlib",
}

def install_if_missing(packages: dict):
    """Install package jika belum ada."""
    for pip_name, import_name in packages.items():
        try:
            importlib.import_module(import_name)
            print(f"  [OK]      {pip_name}")
        except ImportError:
            print(f"  [INSTALL] Menginstal {pip_name} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])
            print(f"  [DONE]    {pip_name} berhasil diinstal")

print("=" * 55)
print("  Memeriksa dependensi ...")
print("=" * 55)
install_if_missing(REQUIRED_PACKAGES)
print("\nSemua dependensi siap.")


  Memeriksa dependensi ...
  [OK]      xgboost
  [OK]      PySastrawi
  [OK]      nltk
  [OK]      scikit-learn


d:\freecodecamp\mbg_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [OK]      mlflow
  [OK]      imbalanced-learn
  [OK]      wordcloud
  [OK]      seaborn
  [OK]      matplotlib

Semua dependensi siap.


## 1. Import Library Utama

In [2]:
import os
import sys
import re
import time
import logging
import warnings
import pickle
import json
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend untuk server/CI
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# Tambahkan root project ke sys.path agar modul lokal bisa diimport
sys.path.insert(0, os.getcwd())

from config import (
    DATA_PATH, OUTPUT_DIR, MODEL_DIR, LOG_DIR,
    MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT, MLFLOW_RUN_NAME,
    XGB_PARAMS, TEST_SIZE, CV_FOLDS, RANDOM_SEED,
)
from src.preprocessor import IndonesianPreprocessor
from src.labeler import SentimentLabeler
from src.feature_engineering import FeatureBuilder

warnings.filterwarnings("ignore")
print("Import selesai.")


Import selesai.


## 2. Setup Logging

In [3]:
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

log_path = os.path.join(LOG_DIR, "train.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("train")
logger.info("Logging aktif → %s", log_path)


2026-05-22 23:48:45,953 | INFO     | train | Logging aktif → d:\freecodecamp\mbg_project\logs\train.log


## 3. Fungsi Visualisasi

In [ ]:
def plot_eda(df: pd.DataFrame) -> str:
    
    logger.info("Membuat grafik EDA ...")
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("EDA — Komentar YouTube Program MBG", fontsize=14, fontweight="bold")

    axes[0, 0].hist(df["text_len"], bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
    axes[0, 0].set_title("Distribusi Panjang Komentar")
    axes[0, 0].set_xlabel("Jumlah Karakter")
    axes[0, 0].set_ylabel("Frekuensi")

    axes[0, 1].hist(df["word_count"], bins=40, color="#DD8452", edgecolor="white", alpha=0.85)
    axes[0, 1].set_title("Distribusi Jumlah Kata")
    axes[0, 1].set_xlabel("Jumlah Kata")

    all_words = " ".join(df["text"].astype(str)).lower().split()
    top_20    = Counter(all_words).most_common(20)
    words, cnt = zip(*top_20)
    axes[1, 0].barh(words[::-1], cnt[::-1], color="#55A868")
    axes[1, 0].set_title("Top 20 Kata (Sebelum Preprocessing)")
    axes[1, 0].set_xlabel("Frekuensi")

    axes[1, 1].hist(df["like_count"].clip(upper=50), bins=30, color="#C44E52", edgecolor="white", alpha=0.85)
    axes[1, 1].set_title("Distribusi Like Count (Clip @50)")
    axes[1, 1].set_xlabel("Like Count")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "1_eda.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def plot_label_distribution(df: pd.DataFrame) -> str:
    logger.info("Membuat grafik distribusi label ...")
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Distribusi Label — Sentimen & Aspek", fontsize=13, fontweight="bold")

    sent_color  = {"positif": "#2ecc71", "negatif": "#e74c3c", "netral": "#f39c12"}
    sent_counts = df["sentimen"].value_counts()

    bars = axes[0].bar(
        sent_counts.index, sent_counts.values,
        color=[sent_color.get(s, "#aaa") for s in sent_counts.index],
        edgecolor="white",
    )
    axes[0].set_title("Distribusi Sentimen")
    axes[0].set_ylabel("Jumlah")
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width() / 2, h + 20, str(int(h)), ha="center", fontweight="bold")

    asp_counts = df["aspek"].value_counts()
    axes[1].barh(asp_counts.index, asp_counts.values, color="#3498db", edgecolor="white")
    axes[1].set_title("Distribusi Aspek")
    axes[1].set_xlabel("Jumlah")

    axes[2].pie(
        sent_counts.values,
        labels=sent_counts.index,
        colors=[sent_color.get(s, "#aaa") for s in sent_counts.index],
        autopct="%1.1f%%", startangle=90,
    )
    axes[2].set_title("Proporsi Sentimen (%)")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "2_label_distribution.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def plot_evaluation(y_test, y_pred, cv_scores, class_names, acc, f1) -> str:
    logger.info("Membuat grafik evaluasi ...")
    cm = confusion_matrix(y_test, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Evaluasi Model — Attention-Enhanced XGBoost\n"
        f"Akurasi: {acc*100:.2f}%  |  F1: {f1*100:.2f}%",
        fontsize=12, fontweight="bold"
    )

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.5)
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Prediksi")
    axes[0].set_ylabel("Aktual")

    fold_labels = [f"Fold {i+1}" for i in range(len(cv_scores))]
    axes[1].bar(fold_labels, cv_scores * 100, color="#3498db", edgecolor="white")
    axes[1].axhline(y=cv_scores.mean() * 100, color="red",
                    linestyle="--", label=f"Mean: {cv_scores.mean()*100:.2f}%")
    axes[1].set_ylim(50, 105)
    axes[1].set_title("5-Fold Cross Validation Accuracy")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    for i, v in enumerate(cv_scores):
        axes[1].text(i, v * 100 + 0.5, f"{v*100:.1f}%", ha="center", fontsize=9)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "3_evaluasi.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def plot_absa(df: pd.DataFrame) -> str:
    logger.info("Membuat grafik ABSA ...")
    absa = df.groupby(["aspek", "sentimen"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 6))
    absa.plot(kind="bar", ax=ax,
              color={"negatif": "#e74c3c", "netral": "#f39c12", "positif": "#2ecc71"},
              edgecolor="white", width=0.7)
    ax.set_title(
        "Aspect-Based Sentiment Analysis — Program MBG\n(Distribusi Sentimen per Aspek)",
        fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Aspek")
    ax.set_ylabel("Jumlah Komentar")
    ax.legend(title="Sentimen")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "4_absa_per_aspek.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def plot_wordcloud(df: pd.DataFrame) -> str:
    try:
        from wordcloud import WordCloud
    except ImportError:
        logger.warning("wordcloud tidak tersedia, skip.")
        return None

    logger.info("Membuat WordCloud ...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("WordCloud per Sentimen", fontsize=12, fontweight="bold")

    cmap_dict = {"positif": "Greens", "negatif": "Reds", "netral": "Oranges"}
    for ax, sent in zip(axes, ["positif", "negatif", "netral"]):
        corpus = " ".join(df[df["sentimen"] == sent]["text_clean"].dropna())
        if not corpus.strip():
            ax.set_visible(False)
            continue
        wc = WordCloud(width=500, height=300, background_color="white",
                       colormap=cmap_dict[sent], max_words=80).generate(corpus)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        n = (df["sentimen"] == sent).sum()
        ax.set_title(f"Sentimen: {sent.capitalize()} ({n:,} komentar)")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "5_wordcloud.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path

print("Semua fungsi visualisasi siap.")


Semua fungsi visualisasi siap.


## 4. STEP 1 — Load Dataset

In [ ]:
# ── Konfigurasi
SAMPLE_SIZE = None   

# ── Load CSV 
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}\n"
        "Pastikan file CSV ada di direktori yang sama."
    )

df = pd.read_csv(DATA_PATH)
logger.info("Data dimuat: %d baris, %d kolom", len(df), len(df.columns))

if SAMPLE_SIZE:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED)
    logger.info("Mode sample: %d baris", len(df))

df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"]       = df["text"].astype(str)
df["text_len"]   = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

logger.info("Dataset final: %d baris", len(df))
df.head()


2026-05-22 23:50:45,761 | INFO     | train | Data dimuat: 10999 baris, 5 kolom
2026-05-22 23:50:45,801 | INFO     | train | Dataset final: 10998 baris


,video_id,author,text,like_count,published_at,text_len,word_count
0,0BRFlBVWH3c,@asshanumislamvideo3105,11:00 BEGITU MULIA BAPAK...😢\nWajah Murid-muri...,0,2026-04-20T01:05:30Z,112,16
1,0BRFlBVWH3c,@SafariFantasy,Masa jambu biji mentah.pisang mentah suruh mak...,0,2026-04-19T15:48:58Z,63,10
2,0BRFlBVWH3c,@SafariFantasy,Dari adanya mbg yang kurang bagus penyajian da...,0,2026-04-19T15:47:48Z,157,25
3,0BRFlBVWH3c,@jotinambunan2267,mbg: makanan bergizi gratis\nmgb: memb*n*h gen...,0,2026-04-19T14:17:22Z,57,8
4,0BRFlBVWH3c,@mellaaputri,Sepanjang nntn istighfar yaallah 😥,0,2026-04-19T01:11:52Z,34,5


## 5. STEP 2 — Exploratory Data Analysis (EDA)

In [6]:
logger.info("STEP 2: EXPLORATORY DATA ANALYSIS")
logger.info("  Rata-rata panjang  : %.1f karakter", df["text_len"].mean())
logger.info("  Rata-rata kata     : %.1f kata",     df["word_count"].mean())
logger.info("  Komentar terpanjang: %d karakter",   df["text_len"].max())
logger.info("  Like count max     : %d",            df["like_count"].max())

eda_path = plot_eda(df)
print(f"\nGrafik disimpan → {eda_path}")


2026-05-22 23:53:39,931 | INFO     | train | STEP 2: EXPLORATORY DATA ANALYSIS


2026-05-22 23:53:39,984 | INFO     | train |   Rata-rata panjang  : 116.5 karakter
2026-05-22 23:53:40,000 | INFO     | train |   Rata-rata kata     : 17.9 kata
2026-05-22 23:53:40,004 | INFO     | train |   Komentar terpanjang: 3579 karakter
2026-05-22 23:53:40,009 | INFO     | train |   Like count max     : 2431
2026-05-22 23:53:40,011 | INFO     | train | Membuat grafik EDA ...

Grafik disimpan → d:\freecodecamp\mbg_project\output\1_eda.png


## 6. STEP 3 — Preprocessing Teks (Bahasa Indonesia)

In [ ]:
logger.info("STEP 3: PREPROCESSING ...")
t0 = time.time()

preprocessor     = IndonesianPreprocessor()
df["text_clean"] = preprocessor.transform_batch(df["text"].tolist())

df = df[df["text_clean"].str.strip().ne("")].reset_index(drop=True)
logger.info("Preprocessing selesai (%.1fs). Sisa data: %d baris", time.time() - t0, len(df))


df[["text", "text_clean"]].head(5)


2026-05-22 23:53:44,796 | INFO     | train | STEP 3: PREPROCESSING ...
2026-05-22 23:53:45,056 | INFO     | src.preprocessor | Preprocessor siap. Stopword: 827 kata, Stemmer: Sastrawi
2026-05-22 23:53:45,729 | INFO     | src.preprocessor | Preprocessing: 1000/10998 selesai
2026-05-22 23:53:46,224 | INFO     | src.preprocessor | Preprocessing: 2000/10998 selesai
2026-05-22 23:53:46,645 | INFO     | src.preprocessor | Preprocessing: 3000/10998 selesai
2026-05-22 23:53:47,012 | INFO     | src.preprocessor | Preprocessing: 4000/10998 selesai
2026-05-22 23:53:47,378 | INFO     | src.preprocessor | Preprocessing: 5000/10998 selesai
2026-05-22 23:53:47,670 | INFO     | src.preprocessor | Preprocessing: 6000/10998 selesai
2026-05-22 23:53:48,019 | INFO     | src.preprocessor | Preprocessing: 7000/10998 selesai
2026-05-22 23:53:48,341 | INFO     | src.preprocessor | Preprocessing: 8000/10998 selesai
2026-05-22 23:53:48,641 | INFO     | src.preprocessor | Preprocessing: 9000/10998 selesai
2026-0

,text,text_clean
0,11:00 BEGITU MULIA BAPAK...😢\nWajah Murid-muri...,mulia wajah murid murid nampak semangat ajar n...
1,Masa jambu biji mentah.pisang mentah suruh mak...,jambu biji mentah pisang mentah suruh makan
2,Dari adanya mbg yang kurang bagus penyajian da...,makan gizi gratis kurang bagus saji perhati gi...
3,mbg: makanan bergizi gratis\nmgb: memb*n*h gen...,makan gizi gratis makan gizi gratis makan gizi...
4,Sepanjang nntn istighfar yaallah 😥,nonton istighfar yaallah


## 7. STEP 4 — Pelabelan Sentimen & Aspek

In [8]:
logger.info("STEP 4: PELABELAN SENTIMEN & ASPEK ...")
labeler = SentimentLabeler()
df      = labeler.label_dataframe(df)

sent_dist = df["sentimen"].value_counts().to_dict()
logger.info("Distribusi sentimen: %s", sent_dist)

label_path = plot_label_distribution(df)
plot_absa(df)

df[["text", "sentimen", "aspek"]].head(5)


2026-05-22 23:54:23,683 | INFO     | train | STEP 4: PELABELAN SENTIMEN & ASPEK ...
2026-05-22 23:54:23,685 | INFO     | src.labeler | Memulai pelabelan sentimen & aspek ...
2026-05-22 23:54:23,819 | INFO     | src.labeler | Distribusi sentimen: {'netral': 4343, 'positif': 3658, 'negatif': 2847}
2026-05-22 23:54:23,821 | INFO     | src.labeler | Distribusi aspek: {'kualitas_makanan': 4851, 'umum': 3306, 'anggaran': 1084, 'program': 931, 'guru_sekolah': 667, 'distribusi': 9}
2026-05-22 23:54:23,824 | INFO     | train | Distribusi sentimen: {'netral': 4343, 'positif': 3658, 'negatif': 2847}
2026-05-22 23:54:23,825 | INFO     | train | Membuat grafik distribusi label ...
2026-05-22 23:54:24,203 | INFO     | train | Membuat grafik ABSA ...


,text,sentimen,aspek
0,11:00 BEGITU MULIA BAPAK...😢\nWajah Murid-muri...,positif,guru_sekolah
1,Masa jambu biji mentah.pisang mentah suruh mak...,negatif,kualitas_makanan
2,Dari adanya mbg yang kurang bagus penyajian da...,negatif,kualitas_makanan
3,mbg: makanan bergizi gratis\nmgb: memb*n*h gen...,positif,kualitas_makanan
4,Sepanjang nntn istighfar yaallah 😥,netral,umum


## 8. STEP 5 — Rekayasa Fitur (TF-IDF + Attention)

In [9]:
logger.info("STEP 5: REKAYASA FITUR ...")
builder = FeatureBuilder()
X, y    = builder.fit_transform(df)
logger.info("Matriks fitur: %s", str(X.shape))
print(f"Jumlah fitur : {X.shape[1]}")
print(f"Jumlah sampel: {X.shape[0]}")


2026-05-22 23:54:28,803 | INFO     | train | STEP 5: REKAYASA FITUR ...
2026-05-22 23:54:28,807 | INFO     | src.feature_engineering | Membangun fitur training ...
2026-05-22 23:54:29,617 | INFO     | src.feature_engineering | TF-IDF shape: (10848, 6000)
2026-05-22 23:54:35,039 | INFO     | src.feature_engineering | Total fitur gabungan: 6012
2026-05-22 23:54:35,084 | INFO     | src.feature_engineering | Kelas label: ['negatif', 'netral', 'positif']
2026-05-22 23:54:35,204 | INFO     | train | Matriks fitur: (10848, 6012)
Jumlah fitur : 6012
Jumlah sampel: 10848


## 9. STEP 6 — Train-Test Split & SMOTE

In [10]:
logger.info("STEP 6: TRAIN-TEST SPLIT + SMOTE ...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

# SMOTE hanya pada data training
smote = SMOTE(random_state=RANDOM_SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

bal_dist = Counter(y_train_bal)
logger.info("Data train setelah SMOTE: %d baris", len(y_train_bal))
logger.info("Distribusi kelas: %s",
    {builder.le_label.classes_[k]: v for k, v in bal_dist.items()})

print(f"\nUkuran data training (setelah SMOTE): {X_train_bal.shape}")
print(f"Ukuran data test                     : {X_test.shape}")


2026-05-22 23:54:35,275 | INFO     | train | STEP 6: TRAIN-TEST SPLIT + SMOTE ...


2026-05-22 23:54:46,151 | INFO     | train | Data train setelah SMOTE: 10422 baris
2026-05-22 23:54:46,173 | INFO     | train | Distribusi kelas: {'positif': 3474, 'netral': 3474, 'negatif': 3474}

Ukuran data training (setelah SMOTE): (10422, 6012)
Ukuran data test                     : (2170, 6012)


## 10. STEP 7 — Training Attention-Enhanced XGBoost

In [11]:
logger.info("STEP 7: TRAINING ATTENTION-ENHANCED XGBOOST ...")
logger.info("Parameter XGBoost: %s", XGB_PARAMS)

sample_w = compute_sample_weight("balanced", y_train_bal)

model = XGBClassifier(**XGB_PARAMS)
t0    = time.time()
model.fit(X_train_bal, y_train_bal, sample_weight=sample_w, verbose=False)
train_time = time.time() - t0

logger.info("Training selesai dalam %.1f detik", train_time)
print(f"\nTraining selesai: {train_time:.1f} detik")


2026-05-22 23:54:46,216 | INFO     | train | STEP 7: TRAINING ATTENTION-ENHANCED XGBOOST ...
2026-05-22 23:54:46,219 | INFO     | train | Parameter XGBoost: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.08, 'subsample': 0.85, 'colsample_bytree': 0.75, 'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 1.5, 'eval_metric': 'mlogloss', 'random_state': 42, 'n_jobs': 1, 'tree_method': 'hist'}
2026-05-23 00:04:17,973 | INFO     | train | Training selesai dalam 571.7 detik

Training selesai: 571.7 detik


## 11. STEP 8 — Evaluasi Model

In [12]:
logger.info("STEP 8: EVALUASI MODEL ...")
y_pred = model.predict(X_test)

acc       = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted")
recall    = recall_score(y_test, y_pred, average="weighted")

print("=" * 42)
print(f"  AKURASI TEST SET : {acc*100:.2f}%")
print(f"  F1   (weighted)  : {f1*100:.2f}%")
print(f"  PRECISION        : {precision*100:.2f}%")
print(f"  RECALL           : {recall*100:.2f}%")
print("=" * 42)

class_names = builder.le_label.classes_
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Cross-Validation
logger.info("Cross-Validation (%d fold) ...", CV_FOLDS)
cv_model  = XGBClassifier(**XGB_PARAMS)
cv_scores = cross_val_score(cv_model, X, y, cv=CV_FOLDS, scoring="accuracy", n_jobs=-1)
logger.info("CV Accuracy: %.2f%% ± %.2f%%", cv_scores.mean()*100, cv_scores.std()*100)

eval_path = plot_evaluation(y_test, y_pred, cv_scores, class_names, acc, f1)
wc_path   = plot_wordcloud(df)


2026-05-23 00:04:18,353 | INFO     | train | STEP 8: EVALUASI MODEL ...
  AKURASI TEST SET : 96.18%
  F1   (weighted)  : 96.17%
  PRECISION        : 96.18%
  RECALL           : 96.18%

Classification Report:

              precision    recall  f1-score   support

     negatif     0.9624    0.9455    0.9539       569
      netral     0.9599    0.9643    0.9621       869
     positif     0.9634    0.9713    0.9673       732

    accuracy                         0.9618      2170
   macro avg     0.9619    0.9604    0.9611      2170
weighted avg     0.9618    0.9618    0.9617      2170

2026-05-23 00:04:18,954 | INFO     | train | Cross-Validation (5 fold) ...
2026-05-23 00:15:20,185 | INFO     | train | CV Accuracy: 96.14% ± 0.90%
2026-05-23 00:15:20,255 | INFO     | train | Membuat grafik evaluasi ...
2026-05-23 00:15:21,545 | INFO     | train | Membuat WordCloud ...


## 12. STEP 9 — Simpan Model & Artefak

In [13]:
logger.info("STEP 9: MENYIMPAN MODEL & ARTEFAK ...")

# Simpan model XGBoost
model_path = os.path.join(MODEL_DIR, "xgboost_model.json")
model.save_model(model_path)

# Simpan FeatureBuilder
builder.save()

# Simpan LabelEncoder
le_path = os.path.join(MODEL_DIR, "label_encoder.pkl")
with open(le_path, "wb") as f:
    pickle.dump(builder.le_label, f)

# Metadata model
metadata = {
    "accuracy"    : round(acc, 4),
    "f1_weighted" : round(f1, 4),
    "precision"   : round(precision, 4),
    "recall"      : round(recall, 4),
    "cv_mean"     : round(cv_scores.mean(), 4),
    "cv_std"      : round(cv_scores.std(), 4),
    "n_samples"   : len(df),
    "n_features"  : X.shape[1],
    "classes"     : list(class_names),
    "train_time_s": round(train_time, 2),
    "xgb_params"  : XGB_PARAMS,
}
meta_path = os.path.join(MODEL_DIR, "model_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Simpan dataset hasil analisis
result_df = df[["video_id", "author", "text", "text_clean",
                 "sentimen", "aspek", "like_count", "published_at"]]
result_path = os.path.join(OUTPUT_DIR, "hasil_sentimen_absa.csv")
result_df.to_csv(result_path, index=False, encoding="utf-8-sig")

logger.info("Model disimpan → %s", model_path)
logger.info("Hasil CSV     → %s", result_path)
print("\nSemua artefak berhasil disimpan.")
print(json.dumps(metadata, indent=2, ensure_ascii=False))


2026-05-23 00:15:28,341 | INFO     | train | STEP 9: MENYIMPAN MODEL & ARTEFAK ...
2026-05-23 00:15:28,455 | INFO     | src.feature_engineering | FeatureBuilder disimpan → d:\freecodecamp\mbg_project\models\feature_builder.pkl
2026-05-23 00:15:28,595 | INFO     | train | Model disimpan → d:\freecodecamp\mbg_project\models\xgboost_model.json
2026-05-23 00:15:28,596 | INFO     | train | Hasil CSV     → d:\freecodecamp\mbg_project\output\hasil_sentimen_absa.csv

Semua artefak berhasil disimpan.
{
  "accuracy": 0.9618,
  "f1_weighted": 0.9617,
  "precision": 0.9618,
  "recall": 0.9618,
  "cv_mean": 0.9614,
  "cv_std": 0.009,
  "n_samples": 10848,
  "n_features": 6012,
  "classes": [
    "negatif",
    "netral",
    "positif"
  ],
  "train_time_s": 571.71,
  "xgb_params": {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.08,
    "subsample": 0.85,
    "colsample_bytree": 0.75,
    "min_child_weight": 3,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.5,
    "

## 13. STEP 10 — MLflow Tracking

> Jalankan cell ini jika ingin log ke MLflow. Setelah selesai, buka terminal dan jalankan `mlflow ui` lalu akses http://localhost:5000

In [14]:
USE_MLFLOW = True   # ubah ke False untuk skip MLflow

if USE_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as run:
        logger.info("MLflow Run ID: %s", run.info.run_id)

        # Log hyperparameter
        for key, val in XGB_PARAMS.items():
            mlflow.log_param(key, val)
        mlflow.log_param("test_size",     TEST_SIZE)
        mlflow.log_param("cv_folds",      CV_FOLDS)
        mlflow.log_param("random_seed",   RANDOM_SEED)
        mlflow.log_param("n_samples",     metadata["n_samples"])
        mlflow.log_param("n_features",    metadata["n_features"])
        mlflow.log_param("smote_enabled", True)

        # Log metrik
        mlflow.log_metric("accuracy_test",    metadata["accuracy"])
        mlflow.log_metric("f1_weighted",      metadata["f1_weighted"])
        mlflow.log_metric("precision",        metadata["precision"])
        mlflow.log_metric("recall",           metadata["recall"])
        mlflow.log_metric("cv_mean_accuracy", metadata["cv_mean"])
        mlflow.log_metric("cv_std",           metadata["cv_std"])
        mlflow.log_metric("train_time_s",     metadata["train_time_s"])
        for i, score in enumerate(cv_scores, start=1):
            mlflow.log_metric(f"cv_fold_{i}", float(score))

        # Log artefak
        artifacts = {
            "eda"            : eda_path,
            "label_dist"     : label_path,
            "evaluasi"       : eval_path,
            "wordcloud"      : wc_path,
            "model_json"     : model_path,
            "feature_builder": os.path.join(MODEL_DIR, "feature_builder.pkl"),
            "label_encoder"  : le_path,
            "metadata"       : meta_path,
            "hasil_csv"      : result_path,
        }
        for name, path in artifacts.items():
            if path and os.path.exists(path):
                mlflow.log_artifact(path)
                logger.info("  Artefak di-log: %s", os.path.basename(path))

        # Log model ke registry
        try:
            from mlflow.models import infer_signature
            sample_input = np.zeros((1, metadata["n_features"]))
            signature    = infer_signature(sample_input, model.predict(sample_input))
            mlflow.xgboost.log_model(
                xgb_model=model,
                artifact_path="xgboost_model",
                signature=signature,
                registered_model_name="ABSA_MBG_XGBoost",
            )
            logger.info("Model di-log ke MLflow Model Registry")
        except Exception as e:
            logger.warning("Gagal log model ke registry: %s", e)
            mlflow.log_artifact(model_path)

        print(f"\nMLflow logging selesai. Run ID: {run.info.run_id}")
        print("Jalankan: mlflow ui")
        print("Lalu buka: http://localhost:5000")
else:
    print("MLflow tracking dinonaktifkan (USE_MLFLOW = False)")


2026-05-23 00:15:29,709 | INFO     | train | MLflow Run ID: 675ba258e1da4b3387c928e422e7b20f
2026-05-23 00:15:30,162 | INFO     | train |   Artefak di-log: 1_eda.png
2026-05-23 00:15:30,168 | INFO     | train |   Artefak di-log: 2_label_distribution.png
2026-05-23 00:15:30,175 | INFO     | train |   Artefak di-log: 3_evaluasi.png
2026-05-23 00:15:30,186 | INFO     | train |   Artefak di-log: 5_wordcloud.png
2026-05-23 00:15:30,194 | INFO     | train |   Artefak di-log: xgboost_model.json
2026-05-23 00:15:30,202 | INFO     | train |   Artefak di-log: feature_builder.pkl
2026-05-23 00:15:30,208 | INFO     | train |   Artefak di-log: label_encoder.pkl
2026-05-23 00:15:30,216 | INFO     | train |   Artefak di-log: model_metadata.json
2026-05-23 00:15:30,225 | INFO     | train |   Artefak di-log: hasil_sentimen_absa.csv


2026/05/23 00:15:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'ABSA_MBG_XGBoost' already exists. Creating a new version of this model...
Created version '4' of model 'ABSA_MBG_XGBoost'.


2026-05-23 00:15:52,540 | INFO     | train | Model di-log ke MLflow Model Registry

MLflow logging selesai. Run ID: 675ba258e1da4b3387c928e422e7b20f
Jalankan: mlflow ui
Lalu buka: http://localhost:5000


## 14. Ringkasan Hasil Training

In [15]:
print("=" * 55)
print("  RINGKASAN HASIL TRAINING")
print("=" * 55)
print(f"  Dataset             : {len(df):,} komentar")
print(f"  Akurasi Test Set    : {acc*100:.2f}%")
print(f"  F1-Score (weighted) : {f1*100:.2f}%")
print(f"  CV Mean Accuracy    : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print(f"  Output folder       : {OUTPUT_DIR}")
print(f"  Model folder        : {MODEL_DIR}")
print("=" * 55)
print("  ✅ TRAINING SELESAI!")
print("=" * 55)


  RINGKASAN HASIL TRAINING
  Dataset             : 10,848 komentar
  Akurasi Test Set    : 96.18%
  F1-Score (weighted) : 96.17%
  CV Mean Accuracy    : 96.14% ± 0.90%
  Output folder       : d:\freecodecamp\mbg_project\output
  Model folder        : d:\freecodecamp\mbg_project\models
  ✅ TRAINING SELESAI!
